
# 互信息（MI）特征选择实验

## 一、实验目的

使用互信息方法对 UCI HAR 数据集的 561 个工程特征进行排序，研究减少输入特征数量对人体活动识别性能和计算效率的影响。

## 二、实验设置

- 数据集：UCI HAR
- 训练集：7,352 个样本
- 测试集：2,947 个样本
- 原始特征数量：561
- 特征选择方法：Mutual Information
- 候选特征数量：30、60、100、200
- 分类模型：RBF-SVM
- 分类参数：C = 1.0，gamma = scale
- 基线 Accuracy：95.22%
- 基线 Macro-F1：95.15%

## 三、实验流程

1. 读取官方训练集和测试集。
2. 仅在训练集上计算各特征的互信息得分。
3. 根据得分排序，生成 Top-30、Top-60、Top-100、Top-200 特征集合。
4. 保存特征排名和筛选结果。
5. 使用筛选后的特征重新训练 SVM。
6. 在官方测试集上计算评价指标。
7. 比较不同特征数量下的分类性能和计算效率。




In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
import time

from sklearn.feature_selection import mutual_info_classif

# 确定项目目录
项目根目录 = Path.cwd().parent
数据目录 = 项目根目录 / "data" / "raw" / "UCI HAR Dataset"

# 读取官方训练集
X_train = pd.read_csv(
    数据目录 / "train" / "X_train.txt",
    sep=r"\s+",
    header=None
)

y_train = pd.read_csv(
    数据目录 / "train" / "y_train.txt",
    sep=r"\s+",
    header=None
).iloc[:, 0]

# 读取官方测试集
X_test = pd.read_csv(
    数据目录 / "test" / "X_test.txt",
    sep=r"\s+",
    header=None
)

y_test = pd.read_csv(
    数据目录 / "test" / "y_test.txt",
    sep=r"\s+",
    header=None
).iloc[:, 0]

# 检查数据是否正确
assert X_train.shape == (7352, 561)
assert X_test.shape == (2947, 561)
assert len(y_train) == len(X_train)
assert len(y_test) == len(X_test)

print("训练集维度：", X_train.shape)
print("测试集维度：", X_test.shape)

# 计算互信息，只使用训练集
互信息开始时间 = time.perf_counter()

互信息得分 = mutual_info_classif(
    X_train,
    y_train,
    discrete_features=False,
    random_state=42,
    n_neighbors=3
)

互信息耗时 = time.perf_counter() - 互信息开始时间

# 建立特征排名表
特征排名表 = pd.DataFrame({
    "特征索引": np.arange(X_train.shape[1]),
    "互信息得分": 互信息得分
})

# 得分由高到低排序
特征排名表 = (
    特征排名表
    .sort_values("互信息得分", ascending=False)
    .reset_index(drop=True)
)

print("互信息计算完成")
print(f"计算耗时：{互信息耗时:.3f} 秒")
print("特征总数：", len(特征排名表))
print("\n互信息得分最高的前 10 个特征：")

display(特征排名表.head(10))


训练集维度： (7352, 561)
测试集维度： (2947, 561)
互信息计算完成
计算耗时：13.875 秒
特征总数： 561

互信息得分最高的前 10 个特征：


,特征索引,互信息得分
0,9,1.006893
1,53,0.944219
2,89,0.937857
3,50,0.928346
4,203,0.904677
5,216,0.904643
6,90,0.892837
7,268,0.890931
8,310,0.890793
9,302,0.887700


In [2]:

# 定义要比较的特征数量
特征数量列表 = [30, 60, 100, 200]

# 创建结果目录
结果目录 = 项目根目录 / "results"
结果目录.mkdir(parents=True, exist_ok=True)

# 保存完整特征排名
排名文件 = 结果目录 / "mi_feature_ranking.csv"
特征排名表.to_csv(
    排名文件,
    index=False,
    encoding="utf-8-sig"
)

print("完整特征排名已保存到：", 排名文件)

# 生成不同规模的特征索引
MI特征索引 = {}

for k in 特征数量列表:
    MI特征索引[k] = (
        特征排名表
        .head(k)["特征索引"]
        .to_numpy(dtype=int)
    )

    assert len(MI特征索引[k]) == k
    assert len(np.unique(MI特征索引[k])) == k

    print(f"Top-{k}：已选择 {len(MI特征索引[k])} 个特征")

# 保存不同规模的特征索引
索引记录 = []

for k, 索引数组 in MI特征索引.items():
    for 排名位置, 特征索引 in enumerate(索引数组, start=1):
        索引记录.append({
            "特征数量": k,
            "排名": 排名位置,
            "特征索引": int(特征索引),
            "互信息得分": float(互信息得分[特征索引])
        })

MI索引表 = pd.DataFrame(索引记录)

索引文件 = 结果目录 / "mi_selected_features.csv"
MI索引表.to_csv(
    索引文件,
    index=False,
    encoding="utf-8-sig"
)

print("特征索引已保存到：", 索引文件)
display(MI索引表.groupby("特征数量").size())


完整特征排名已保存到： d:\smartphone-feature-selection\results\mi_feature_ranking.csv
Top-30：已选择 30 个特征
Top-60：已选择 60 个特征
Top-100：已选择 100 个特征
Top-200：已选择 200 个特征
特征索引已保存到： d:\smartphone-feature-selection\results\mi_selected_features.csv


特征数量
30      30
60      60
100    100
200    200
dtype: int64